# WDMamba Phase 2: Lightweight RGB baseline (B1)

This notebook is self-contained for Google Colab. It clones WDMamba, downloads RESIDE-6K, searches at most three lightweight RGB configurations, trains the selected configuration on the complete train split, evaluates development/test quality, measures latency/VRAM with the Phase 1 protocol, writes failure-case images and a Pareto plot, and produces `P2_lightweight_report.md` plus a ZIP.

The notebook does not add ASM, wavelet branches, or physics losses. Gate G2 requires at least 20% latency reduction, ΔPSNR >= -0.3 dB versus the full Phase 1 baseline, and manual artifact review.

## 1. Clone and configure

The full Phase 2 run is the default. `SEARCH_ITERS` is used for the three-config dev search and `FINAL_ITERS` for the selected configuration on the full train split. `P1_SUMMARY_PATH` must point to the final full-split Phase 1 `summary.csv` before Gate G2 can pass.

In [ ]:
from pathlib import Path
import json, platform, subprocess, sys

REPO_URL = "https://github.com/Darkin72/WDMamba.git"
REPO_ROOT = Path("/content/WDMamba_phase2")
WORK_ROOT = Path("/content/wdmamba_phase2")
OUTPUT_ROOT = WORK_ROOT / "results"
ENV_ROOT = Path("/content/wdmamba_phase2_python")
PYTHON = ENV_ROOT / "bin/python"
if platform.system() != "Linux" or not Path("/content").is_dir():
    raise RuntimeError("This notebook is intended for Google Colab/Linux.")

def run_command(args, cwd=None, log_path=None):
    args = [str(x) for x in args]
    print("$", " ".join(args), flush=True)
    if log_path is None:
        log_path = WORK_ROOT / "logs" / "command.log"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True); log.write(line); log.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command failed ({rc}); full log: {log_path}")

if not REPO_ROOT.exists():
    run_command(["git", "clone", "--depth", "1", REPO_URL, REPO_ROOT])
run_command(["git", "log", "-1", "--oneline"], cwd=REPO_ROOT)

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16  # overwritten from Phase 1 summary.csv
PATCH_SIZE = 256
MAX_SIDE = 0  # overwritten from Phase 1 summary.csv
SEARCH_ITERS = 2000
FINAL_ITERS = 10000
SEARCH_DEV_LIMIT = 256
TEST_LIMIT = 0
NUM_WORKERS = 2
WARMUP = 10
TIMING_REPEATS = 30
SEED = 666
ARTIFACTS_REVIEWED = False
P1_BASELINE_ACCEPTED = False  # Set True only after explaining/accepting the Phase 1 paper mismatch.
P1_SUMMARY_PATH = ""  # Upload/copy the final full-split Phase 1 summary.csv here.
KAGGLE_DATASET = "kmljts/reside-6k"
SEARCH_CONFIGS = [
    {"name": "B1-small", "base": 16, "blocks": 4},
    {"name": "B1-medium", "base": 24, "blocks": 6},
    {"name": "B1-large", "base": 32, "blocks": 6},
]
CONFIG = {}
CONFIG.update({"repo_root": str(REPO_ROOT), "work_root": str(WORK_ROOT), "output_root": str(OUTPUT_ROOT),
               "train_batch_size": TRAIN_BATCH_SIZE, "eval_batch_size": EVAL_BATCH_SIZE,
               "patch_size": PATCH_SIZE, "max_side": MAX_SIDE,
               "search_iters": SEARCH_ITERS, "final_iters": FINAL_ITERS,
               "search_dev_limit": SEARCH_DEV_LIMIT, "test_limit": TEST_LIMIT,
               "num_workers": NUM_WORKERS, "warmup": WARMUP, "timing_repeats": TIMING_REPEATS,
               "seed": SEED, "artifacts_reviewed": ARTIFACTS_REVIEWED,
               "p1_baseline_accepted": P1_BASELINE_ACCEPTED,
               "p1_summary_path": P1_SUMMARY_PATH, "kaggle_dataset": KAGGLE_DATASET,
               "search_configs": SEARCH_CONFIGS})
WORK_ROOT.mkdir(parents=True, exist_ok=True); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = WORK_ROOT / "config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2, default=str), encoding="utf-8")
print(json.dumps(CONFIG, indent=2, default=str))

### Upload the final Phase 1 baseline

Upload the full-split Phase 1 `summary.csv`. This is mandatory for the Phase 2 speedup and ΔPSNR gate.


In [ ]:
from google.colab import files
import pandas as pd
if not P1_SUMMARY_PATH or not Path(P1_SUMMARY_PATH).is_file():
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("Upload exactly one final Phase 1 summary.csv file.")
    baseline_target = WORK_ROOT / "P1_full_summary.csv"
    baseline_target.write_bytes(uploaded[csv_names[0]])
    P1_SUMMARY_PATH = str(baseline_target)
baseline = pd.read_csv(P1_SUMMARY_PATH)
required = {"configured_batch_size", "max_side", "PSNR", "SSIM", "original_forward_ms_per_image"}
missing = required - set(baseline.columns)
if missing:
    raise ValueError(f"Phase 1 summary is missing protocol columns: {sorted(missing)}")
EVAL_BATCH_SIZE = int(baseline["configured_batch_size"].iloc[0])
MAX_SIDE = int(baseline["max_side"].iloc[0])
CONFIG.update({"p1_summary_path": P1_SUMMARY_PATH, "eval_batch_size": EVAL_BATCH_SIZE, "max_side": MAX_SIDE})
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2, default=str), encoding="utf-8")
print("Phase 1 baseline:", P1_SUMMARY_PATH)
print("Inherited evaluation protocol: batch_size=", EVAL_BATCH_SIZE, "MAX_SIDE=", MAX_SIDE)


## 2. Install the isolated training environment

In [ ]:
run_command([sys.executable, "-m", "pip", "install", "-q", "uv"])
run_command([sys.executable, "-m", "uv", "python", "install", "3.11"])
if not PYTHON.is_file():
    run_command([sys.executable, "-m", "uv", "venv", "--python", "3.11", ENV_ROOT])
run_command([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON,
             "--index-url", "https://download.pytorch.org/whl/cu128", "torch==2.8.0", "torchvision==0.23.0"])
constraints = ENV_ROOT / "constraints.txt"
constraints.write_text("torch==2.8.0\ntorchvision==0.23.0\nnumpy<2\nsetuptools<81\n", encoding="utf-8")
run_command([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON, "--constraint", constraints,
             "numpy<2", "opencv-python-headless<4.12", "scipy", "scikit-image", "pandas", "matplotlib", "tqdm", "kagglehub", "pyyaml"])

## 3. Download and resolve train/test pairs

The resolver prefers directories marked `train` and `test`, supports `GT/clear/clean` and `hazy/input` naming, and records the exact paths in `inputs.json`.

In [ ]:
LOCAL_DATA_ROOT = ""
CONFIG["local_data_root"] = LOCAL_DATA_ROOT
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2, default=str), encoding="utf-8")
def run_python(label, source):
    script = WORK_ROOT / f"{label}.py"; script.write_text(source.strip() + "\n", encoding="utf-8")
    run_command([PYTHON, "-u", script, CONFIG_PATH], cwd=REPO_ROOT, log_path=WORK_ROOT / "logs" / f"{label}.log")
run_python("prepare_reside6k", r"""
import json, sys
from pathlib import Path
import kagglehub
cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
base = Path(cfg["local_data_root"]) if cfg.get("local_data_root") else Path(kagglehub.dataset_download(cfg["kaggle_dataset"]))
image_ext = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
def pair_dirs(path):
    dirs = {p.name.lower(): p for p in path.iterdir() if p.is_dir()}
    gt = next((dirs[k] for k in ("gt", "clear", "clean", "groundtruth") if k in dirs), None)
    lq = next((dirs[k] for k in ("hazy", "haze", "input", "inputs") if k in dirs), None)
    if gt and lq and any(p.suffix.lower() in image_ext for p in gt.iterdir() if p.is_file()):
        return gt, lq
    return None
candidates = [base, *[p for p in base.rglob("*") if p.is_dir()]]
pairs = []
for path in candidates:
    found = pair_dirs(path)
    if found: pairs.append((path, *found))
if not pairs: raise FileNotFoundError(f"No paired image directories below {base}")
def score(item, split):
    text = "/".join(x.name.lower() for x in item)
    return (0 if split in text else 1, len(text))
train = sorted(pairs, key=lambda x: score(x, "train"))[0]
test = sorted(pairs, key=lambda x: score(x, "test"))[0]
if train[0] == test[0] and len(pairs) > 1: test = sorted(pairs, key=lambda x: score(x, "test"))[1]
manifest = {"dataset_root": str(base.resolve()), "train_root": str(train[0].resolve()), "train_clear": str(train[1].resolve()), "train_hazy": str(train[2].resolve()), "test_root": str(test[0].resolve()), "test_clear": str(test[1].resolve()), "test_hazy": str(test[2].resolve())}
Path(cfg["work_root"], "inputs.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
""")

## 4. Validate and run B1 search/final training

Search evaluates all three configurations on the development subset. The selected configuration is then trained using the complete train split and evaluated on the complete test split.

In [ ]:
import ast, textwrap
TRAINER_SOURCE = textwrap.dedent('import json, math, os, random, re, subprocess, sys, time\nfrom pathlib import Path\nimport cv2, numpy as np, pandas as pd, torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\nfrom torchvision.models import vgg19, VGG19_Weights\nfrom tqdm import tqdm\n\nCFG = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))\nINPUTS = json.loads((Path(CFG["work_root"]) / "inputs.json").read_text(encoding="utf-8"))\nOUT = Path(CFG["output_root"]); OUT.mkdir(parents=True, exist_ok=True)\ntorch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"]); random.seed(CFG["seed"])\ntorch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False\nassert torch.cuda.is_available(), "CUDA GPU required"\nDEVICE = torch.device("cuda")\nEXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}\n\ndef list_images(root): return sorted(p for p in Path(root).iterdir() if p.is_file() and p.suffix.lower() in EXT)\ndef make_pairs(clear_root, hazy_root, limit=0):\n    clear = {p.stem: p for p in list_images(clear_root)}; pairs = []\n    for p in list_images(hazy_root):\n        gt = clear.get(p.stem, clear.get(p.stem.split("_")[0]))\n        if gt: pairs.append((p.stem, p, gt))\n    if limit > 0: pairs = pairs[:limit]\n    if not pairs: raise RuntimeError(f"No pairs found: {clear_root} / {hazy_root}")\n    return pairs\n\nclass PairDataset(Dataset):\n    def __init__(self, pairs, patch=256, train=False): self.pairs, self.patch, self.train = pairs, patch, train\n    def __len__(self): return len(self.pairs)\n    def __getitem__(self, i):\n        name, lq_path, gt_path = self.pairs[i]\n        lq = cv2.cvtColor(cv2.imread(str(lq_path)), cv2.COLOR_BGR2RGB); gt = cv2.cvtColor(cv2.imread(str(gt_path)), cv2.COLOR_BGR2RGB)\n        if lq.shape[:2] != gt.shape[:2]: lq = cv2.resize(lq, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_AREA)\n        if self.train:\n            h, w = gt.shape[:2]; side = min(self.patch, h, w); y = random.randint(0, h-side); x = random.randint(0, w-side); lq, gt = lq[y:y+side, x:x+side], gt[y:y+side, x:x+side]\n            if random.random() < .5: lq, gt = lq[:, ::-1], gt[:, ::-1]\n        else:\n            side = CFG["max_side"]\n            if side > 0 and max(gt.shape[:2]) > side:\n                scale = side / max(gt.shape[:2]); size = (round(gt.shape[1]*scale), round(gt.shape[0]*scale)); lq = cv2.resize(lq, size, interpolation=cv2.INTER_AREA); gt = cv2.resize(gt, size, interpolation=cv2.INTER_AREA)\n        lt = torch.from_numpy(lq.copy()).permute(2,0,1).float()/255.; gt = torch.from_numpy(gt.copy()).permute(2,0,1).float()/255.\n        return lt, gt, name\n\ndef collate(batch):\n    hs = [x[0].shape[-2] for x in batch]; ws = [x[0].shape[-1] for x in batch]; h = math.ceil(max(hs)/32)*32; w = math.ceil(max(ws)/32)*32\n    xs, ys, names, shapes = [], [], [], []\n    for x,y,n in batch:\n        shapes.append(y.shape[-2:]); xs.append(F.pad(x, (0,w-x.shape[-1],0,h-x.shape[-2]), mode="replicate")); ys.append(F.pad(y, (0,w-y.shape[-1],0,h-y.shape[-2]), mode="replicate")); names.append(n)\n    return torch.stack(xs), torch.stack(ys), names, shapes\n\nclass DSBlock(nn.Module):\n    def __init__(self, c):\n        super().__init__(); self.body = nn.Sequential(nn.Conv2d(c,c,3,padding=1,groups=c), nn.Conv2d(c,c,1), nn.GELU(), nn.Conv2d(c,c,3,padding=1,groups=c), nn.Conv2d(c,c,1))\n    def forward(self,x): return x + self.body(x)\nclass RGBLightNet(nn.Module):\n    def __init__(self, base, blocks):\n        super().__init__(); self.head=nn.Conv2d(3,base,3,padding=1); self.body=nn.Sequential(*[DSBlock(base) for _ in range(blocks)]); self.tail=nn.Conv2d(base,3,3,padding=1)\n    def forward(self,x): return (x + self.tail(self.body(self.head(x)))).clamp(0,1)\n\ndef dwt_ll(x): return (x[:,:,0::2,0::2]+x[:,:,1::2,0::2]+x[:,:,0::2,1::2]+x[:,:,1::2,1::2])/2\ndef ssim_loss(x,y):\n    mu_x=F.avg_pool2d(x,3,1,1); mu_y=F.avg_pool2d(y,3,1,1); sx=F.avg_pool2d(x*x,3,1,1)-mu_x*mu_x; sy=F.avg_pool2d(y*y,3,1,1)-mu_y*mu_y; sxy=F.avg_pool2d(x*y,3,1,1)-mu_x*mu_y; c1=.01**2; c2=.03**2\n    s=((2*mu_x*mu_y+c1)*(2*sxy+c2))/((mu_x*mu_x+mu_y*mu_y+c1)*(sx+sy+c2)); return 1-s.mean()\ndef fft_loss(x,y): return (torch.fft.rfft2(x).abs()-torch.fft.rfft2(y).abs()).abs().mean()\nclass VGGContrast(nn.Module):\n    def __init__(self):\n        super().__init__(); self.features=vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features[:30].eval(); self.picks={1,6,11,20,29}; self.weights=[1/32,1/16,1/8,1/4,1]\n        for p in self.parameters(): p.requires_grad=False\n    def encode(self,x):\n        rows=[]\n        for i,layer in enumerate(self.features):\n            x=layer(x)\n            if i in self.picks: rows.append(x)\n        return rows\n    def forward(self,anchor,positive,negative):\n        a=self.encode(anchor); p=self.encode(positive); n=self.encode(negative); return sum(w*F.l1_loss(x,y.detach())/(F.l1_loss(x,z.detach())+1e-7) for w,x,y,z in zip(self.weights,a,p,n))\nCONTRAST=VGGContrast().to(DEVICE)\ndef loss_fn(pred, gt, lq):\n    pixel=F.l1_loss(pred,gt)+.1*ssim_loss(pred,gt); ll=F.l1_loss(dwt_ll(pred),dwt_ll(gt)); freq=.1*fft_loss(pred,gt); contrast=.001*CONTRAST(pred,gt,lq); return pixel+ll+freq+contrast\ndef params(model): return sum(p.numel() for p in model.parameters())\n\ndef train_one(spec, pairs, iters, tag):\n    model=RGBLightNet(spec["base"],spec["blocks"]).to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-3); ds=PairDataset(pairs,CFG["patch_size"],True); loader=DataLoader(ds,batch_size=CFG["train_batch_size"],shuffle=True,num_workers=CFG["num_workers"],pin_memory=True,drop_last=True); it=iter(loader); start=time.perf_counter(); losses=[]\n    model.train()\n    for step in tqdm(range(1,iters+1),desc=tag):\n        try: x,y,_,_=next(it)\n        except StopIteration: it=iter(loader); x,y,_,_=next(it)\n        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True); pred=model(x); loss=loss_fn(pred,y,x); loss.backward(); opt.step(); losses.append(float(loss.detach()))\n    elapsed=time.perf_counter()-start; ckpt=OUT/f"{tag}.pth"; torch.save({"model":model.state_dict(),"spec":spec,"iters":iters,"train_seconds":elapsed},ckpt); return model, {"train_seconds":elapsed,"final_loss":float(np.mean(losses[-100:])),"iters":iters,"checkpoint":str(ckpt)}\n\ndef metrics(model,pairs,tag,save_failures=False):\n    model.eval(); loader=DataLoader(PairDataset(pairs,CFG["patch_size"],False),batch_size=CFG["eval_batch_size"],shuffle=False,num_workers=CFG["num_workers"],pin_memory=True,collate_fn=collate); rows=[]; failures=[]; total_gpu=0.; total_images=0; warm_shapes=set(); torch.cuda.reset_peak_memory_stats()\n    with torch.inference_mode():\n        for x,y,names,shapes in tqdm(loader,desc=f"eval-{tag}"):\n            x=x.to(DEVICE,non_blocking=True)\n            shape_key=tuple(x.shape)\n            if shape_key not in warm_shapes:\n                for _ in range(CFG["warmup"]): model(x)\n                torch.cuda.synchronize(); warm_shapes.add(shape_key)\n            times=[]\n            for _ in range(CFG["timing_repeats"]):\n                t0=torch.cuda.Event(True); t1=torch.cuda.Event(True); t0.record(); pred=model(x); t1.record(); t1.synchronize(); times.append(t0.elapsed_time(t1))\n            total_gpu += float(np.mean(times)); out=pred.cpu().numpy(); gt=y.numpy()\n            for j,(name,shape) in enumerate(zip(names,shapes)):\n                h,w=shape; a=np.clip(out[j,:,:h,:w].transpose(1,2,0)*255,0,255).round().astype(np.uint8); b=np.clip(gt[j,:,:h,:w].transpose(1,2,0)*255,0,255).round().astype(np.uint8); mse=np.mean((a.astype(np.float32)-b.astype(np.float32))**2); psnr=99.0 if mse==0 else 10*np.log10(255**2/mse); from skimage.metrics import structural_similarity; ssim=float(structural_similarity(a,b,channel_axis=-1,data_range=255,gaussian_weights=True,sigma=1.5,use_sample_covariance=False)); rows.append({"model":tag,"image_id":name,"PSNR":float(psnr),"SSIM":ssim}); failures.append((float(psnr),name,a,b))\n            total_images += len(names)\n    failures.sort(key=lambda z:z[0]); fdir=OUT/"failure_cases"/tag; fdir.mkdir(parents=True,exist_ok=True)\n    for _,name,a,b in failures[:8]:\n        cv2.imwrite(str(fdir/f"{name}_pred.png"),cv2.cvtColor(a,cv2.COLOR_RGB2BGR)); cv2.imwrite(str(fdir/f"{name}_gt.png"),cv2.cvtColor(b,cv2.COLOR_RGB2BGR))\n    frame=pd.DataFrame(rows); return {"model":tag,"images":len(frame),"PSNR":float(frame.PSNR.mean()),"SSIM":float(frame.SSIM.mean()),"PSNR_std":float(frame.PSNR.std(ddof=1)),"SSIM_std":float(frame.SSIM.std(ddof=1)),"latency_ms_per_image":total_gpu/total_images,"FPS":1000*total_images/total_gpu,"peak_allocated_GiB":torch.cuda.max_memory_allocated()/1024**3,"params_M":params(model)/1e6}, frame\n\ndef main():\n    train_pairs=make_pairs(INPUTS["train_clear"],INPUTS["train_hazy"]); test_pairs=make_pairs(INPUTS["test_clear"],INPUTS["test_hazy"],CFG["test_limit"]); random.shuffle(train_pairs); dev_pairs=train_pairs[:min(CFG["search_dev_limit"],len(train_pairs))]\n    search_rows=[]; search_ckpts={}\n    for spec in CFG["search_configs"]:\n        m,t=train_one(spec,dev_pairs,CFG["search_iters"],spec["name"]+"_search"); r,_=metrics(m,dev_pairs,spec["name"]+"_dev"); r.update(t); search_rows.append(r); search_ckpts[spec["name"]]=spec\n    pd.DataFrame(search_rows).to_csv(OUT/"search_results.csv",index=False); chosen=min(search_rows,key=lambda r:r["latency_ms_per_image"] if r["PSNR"] >= max(x["PSNR"] for x in search_rows)-.3 else 1e9); chosen_spec=search_ckpts[chosen["model"].split("_")[0]]\n    model,train_info=train_one(chosen_spec,train_pairs,CFG["final_iters"],"B1_selected_final")\n    dev_summary,dev_images=metrics(model,dev_pairs,"B1_selected_dev",False); dev_images.to_csv(OUT/"dev_image_metrics.csv",index=False); pd.DataFrame([dev_summary]).to_csv(OUT/"dev_summary.csv",index=False)\n    test_summary,test_images=metrics(model,test_pairs,"B1_selected_test",True); test_images.to_csv(OUT/"test_image_metrics.csv",index=False); pd.DataFrame([test_summary]).to_csv(OUT/"test_summary.csv",index=False)\n    meta={"gpu":torch.cuda.get_device_name(0),"torch":torch.__version__,"cuda":torch.version.cuda,"train_images":len(train_pairs),"dev_images":len(dev_pairs),"test_images":len(test_pairs),"chosen_config":chosen_spec,"search_results":search_rows,"train_info":train_info,"protocol":f"RGB lightweight B1; same input/output and L1+SSIM+LL+FFT+VGG contrastive loss; train_batch={CFG[\'train_batch_size\']}; eval_batch={CFG[\'eval_batch_size\']}; max_side={CFG[\'max_side\']}"}; (OUT/"metadata.json").write_text(json.dumps(meta,indent=2,default=str),encoding="utf-8")\n    print(pd.DataFrame([test_summary]).to_string(index=False))\nif __name__=="__main__": main()\n').strip() + "\n"
ast.parse(TRAINER_SOURCE)
print("Phase 2 trainer syntax: PASS")


In [ ]:
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2, default=str), encoding="utf-8")
run_python("phase2_b1_runner", TRAINER_SOURCE)

## 5. Report and Gate G2

Set `P1_SUMMARY_PATH` to the final full-split Phase 1 `summary.csv` before running this cell. `ARTIFACTS_REVIEWED` must be set to `True` only after inspecting the saved failure cases.

In [ ]:
import json, shutil
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
out = OUTPUT_ROOT; dev = pd.read_csv(out/"dev_summary.csv"); test = pd.read_csv(out/"test_summary.csv"); search = pd.read_csv(out/"search_results.csv"); dev_row=dev.iloc[0]; row=test.iloc[0]
baseline_ok=False; baseline_psnr=None; baseline_latency=None; protocol_batch=None; protocol_max_side=None
if P1_SUMMARY_PATH and Path(P1_SUMMARY_PATH).is_file():
    b=pd.read_csv(P1_SUMMARY_PATH).iloc[0]; baseline_psnr=float(b["PSNR"]); baseline_latency=float(b["original_forward_ms_per_image"]); protocol_batch=int(b["configured_batch_size"]); protocol_max_side=int(b["max_side"]); baseline_ok=True
delta_psnr=float(row["PSNR"]-baseline_psnr) if baseline_ok else None; speedup=float(baseline_latency/row["latency_ms_per_image"]) if baseline_ok else None
protocol_ok=baseline_ok and int(CONFIG["eval_batch_size"]) == protocol_batch and int(CONFIG["max_side"]) == protocol_max_side
baseline_accepted=bool(P1_BASELINE_ACCEPTED)
quality_ok=baseline_ok and delta_psnr >= -0.3; latency_ok=baseline_ok and speedup >= 1.25; artifact_ok=bool(ARTIFACTS_REVIEWED); gate="PASS" if protocol_ok and baseline_accepted and quality_ok and latency_ok and artifact_ok else "REVIEW"
plt.figure(figsize=(7,5)); plt.scatter(search["latency_ms_per_image"],search["PSNR"],s=70); [plt.annotate(str(x), (y,z)) for x,y,z in zip(search["model"],search["latency_ms_per_image"],search["PSNR"])]; plt.xlabel("Latency (ms/image)"); plt.ylabel("PSNR (dB)"); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(out/"pareto_quality_speed.png",dpi=160); plt.close()
meta=json.loads((out/"metadata.json").read_text(encoding="utf-8")); meta["gate_g2"]={"status":gate,"baseline_available":baseline_ok,"protocol_ok":protocol_ok,"baseline_accepted":baseline_accepted,"delta_psnr":delta_psnr,"speedup":speedup,"quality_ok":quality_ok,"latency_ok":latency_ok,"artifacts_reviewed":artifact_ok}; (out/"metadata.json").write_text(json.dumps(meta,indent=2,default=str),encoding="utf-8")
lines=["# P2 Lightweight RGB Report","","## Gate G2","",f"Status: {gate}","","| Check | Observed | Criterion | Result |","|---|---:|---:|---|",f"| Baseline available | {baseline_ok} | Full Phase 1 summary.csv | {'PASS' if baseline_ok else 'REVIEW'} |",
         f"| Protocol match | batch={protocol_batch}, max_side={protocol_max_side} | inherited from Phase 1 | {'PASS' if protocol_ok else 'REVIEW'} |",
         f"| Phase 1 mismatch accepted | {baseline_accepted} | mismatch explained before Phase 2 | {'PASS' if baseline_accepted else 'REVIEW'} |",f"| Delta PSNR | {delta_psnr if delta_psnr is not None else 'N/A'} | >= -0.300 dB | {'PASS' if quality_ok else 'REVIEW'} |",f"| Speedup | {speedup if speedup is not None else 'N/A'}x | >= 1.25x | {'PASS' if latency_ok else 'REVIEW'} |",f"| Artifact review | {artifact_ok} | Manual review complete | {'PASS' if artifact_ok else 'REVIEW'} |","","## Selected B1","",f"- Development PSNR/SSIM: {dev_row['PSNR']:.4f} / {dev_row['SSIM']:.6f}",f"- Test PSNR: {row['PSNR']:.4f} dB",f"- Test SSIM: {row['SSIM']:.6f}",f"- Latency: {row['latency_ms_per_image']:.4f} ms/image",f"- FPS: {row['FPS']:.4f}",f"- Parameters: {row['params_M']:.4f} M",f"- Peak VRAM: {row['peak_allocated_GiB']:.4f} GiB",f"- Training iterations: {row.get('iters','recorded in metadata')}","","## Required artifacts","","- `search_results.csv`","- `dev_summary.csv`, `dev_image_metrics.csv`, `test_summary.csv`, and `test_image_metrics.csv`","- `pareto_quality_speed.png`","- `failure_cases/`","- `metadata.json`","","## Decision","","Proceed to Phase 3 only if Gate G2 is PASS. Otherwise adjust channels/block depth or retain part of DEN; do not add ASM yet."]
(out/"P2_lightweight_report.md").write_text("\n".join(lines)+"\n",encoding="utf-8"); display(Markdown((out/"P2_lightweight_report.md").read_text(encoding="utf-8")))

## 6. Package Phase 2 results

In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive(str(WORK_ROOT/"wdmamba_phase2_results"),"zip",OUTPUT_ROOT); print("Created:",archive); files.download(archive)